# Fake News Detection Using LSTM Networks and Pre-trained BERT Models
**Esha Hooda (014796990), Tyler Biesemeyer (016488979), Tarif Khan (017050683)**

GitHub: https://github.com/TarifEKhan/Fake-news-detection

Dataset: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

## Cell 1 — Install Dependencies

In [ ]:
!pip install transformers datasets scikit-learn matplotlib seaborn wordcloud kagglehub -q

## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# TensorFlow / Keras (LSTM)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense, Dropout, Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# PyTorch + HuggingFace (BERT)
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW                          # <-- correct location
from transformers import (BertTokenizer,
                           BertForSequenceClassification,
                           get_linear_schedule_with_warmup)

# WordCloud
from wordcloud import WordCloud

print('TF  :', tf.__version__)
print('PT  :', torch.__version__)
print('GPU :', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dev :', device)

## Cell 3 — Load Dataset via kagglehub

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

print('Downloading Fake.csv ...')
df_fake = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'clmentbisaillon/fake-and-real-news-dataset',
    'Fake.csv'
)

print('Downloading True.csv ...')
df_true = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'clmentbisaillon/fake-and-real-news-dataset',
    'True.csv'
)

df_fake['label'] = 0
df_true['label'] = 1

df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Total   : {len(df)}')
print(f'Fake (0): {(df.label==0).sum()}')
print(f'Real (1): {(df.label==1).sum()}')
df.head()

## Cell 4 — EDA

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nLabel counts:')
print(df['label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Label distribution
lc = df['label'].value_counts()
axes[0].bar(['Fake (0)', 'Real (1)'], lc.values, color=['#e74c3c','#2ecc71'], edgecolor='black')
axes[0].set_title('Label Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(lc.values):
    axes[0].text(i, v+100, str(v), ha='center', fontweight='bold')

# Subject distribution
if 'subject' in df.columns:
    sc = df['subject'].value_counts().head(10)
    axes[1].barh(sc.index, sc.values, color='steelblue', edgecolor='black')
    axes[1].set_title('Top 10 Subjects', fontweight='bold')
    axes[1].set_xlabel('Count')
else:
    axes[1].text(0.5, 0.5, 'No subject column', ha='center', va='center')

# Article length
df['text_length'] = df['text'].astype(str).apply(lambda x: len(x.split()))
axes[2].hist(df[df.label==0]['text_length'], bins=50, alpha=0.6, color='#e74c3c', label='Fake')
axes[2].hist(df[df.label==1]['text_length'], bins=50, alpha=0.6, color='#2ecc71', label='Real')
axes[2].set_title('Article Length Distribution', fontweight='bold')
axes[2].set_xlabel('Word Count'); axes[2].set_xlim(0, 2000)
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, label, title, cmap in zip(axes, [0,1],
                                   ['Fake News Word Cloud','Real News Word Cloud'],
                                   ['Reds','Greens']):
    text = ' '.join(df[df.label==label]['text'].astype(str).tolist())
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap=cmap, max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 5 — Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['combined'] = df['title'].astype(str) + ' ' + df['text'].astype(str)
df = df[df['combined'].str.strip() != ''].reset_index(drop=True)

print('Cleaning text... (may take ~1 min)')
df['clean_text'] = df['combined'].apply(clean_text)
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)
print(f'After cleaning: {len(df)} samples')
print('Sample:', df['clean_text'].iloc[0][:200])

## Cell 6 — Sample & Split

In [ ]:
MAX_SAMPLES = 20000
df_f = df[df.label==0].sample(n=min(MAX_SAMPLES//2, (df.label==0).sum()), random_state=42)
df_t = df[df.label==1].sample(n=min(MAX_SAMPLES//2, (df.label==1).sum()), random_state=42)
df_sample = pd.concat([df_f, df_t]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Working set: {len(df_sample)}')
print(df_sample['label'].value_counts())

X = df_sample['clean_text'].values
y = df_sample['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## Cell 7 — Evaluation Helper

In [ ]:
results = []

def evaluate_model(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    print(f'\n{"="*45}')
    print(f'  {name}')
    print(f'{"="*45}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(classification_report(y_true, y_pred, target_names=['Fake','Real']))
    return {'model': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

## Cell 8 — Baseline: SVM + Random Forest

In [ ]:
# SVM
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2))),
    ('clf',   LinearSVC(C=1.0, random_state=42, max_iter=2000))
])
svm_pipeline.fit(X_train, y_train)
svm_preds = svm_pipeline.predict(X_test)
results.append(evaluate_model('SVM (TF-IDF)', y_test, svm_preds))

# Random Forest
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=30000, ngram_range=(1,2))),
    ('clf',   RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)
results.append(evaluate_model('Random Forest (TF-IDF)', y_test, rf_preds))

## Cell 9 — LSTM: Tokenize & Pad

In [ ]:
VOCAB_SIZE    = 60000
MAX_SEQ_LEN   = 400
EMBEDDING_DIM = 128

tokenizer_lstm = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer_lstm.fit_on_texts(X_train)

def texts_to_padded(texts):
    seqs = tokenizer_lstm.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

X_train_pad = texts_to_padded(X_train)
X_val_pad   = texts_to_padded(X_val)
X_test_pad  = texts_to_padded(X_test)

print('Vocab size :', len(tokenizer_lstm.word_index))
print('Train shape:', X_train_pad.shape)

## Cell 10 — LSTM: Build & Train

In [ ]:
def build_lstm_model(vocab_size, embedding_dim, seq_len):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=seq_len),
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm_model(VOCAB_SIZE, EMBEDDING_DIM, MAX_SEQ_LEN)
lstm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

history = lstm_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

## Cell 11 — LSTM: Evaluate & Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train Acc', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Acc',   marker='s')
axes[0].set_title('LSTM Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss',   marker='s')
axes[1].set_title('LSTM Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

lstm_probs = lstm_model.predict(X_test_pad).flatten()
lstm_preds = (lstm_probs >= 0.5).astype(int)
results.append(evaluate_model('BiLSTM', y_test, lstm_preds))

## Cell 12 — BERT: Setup (datasets + model + optimizer)

In [ ]:
# ── BERT hyperparameters ──────────────────────────────────────────
BERT_MODEL_NAME = 'bert-base-uncased'
BERT_MAX_LEN    = 256
BERT_BATCH_SIZE = 16
BERT_EPOCHS     = 3
BERT_LR         = 2e-5

# Subset sizes (keeps Colab training time reasonable)
BERT_TRAIN_SIZE = 8000
BERT_VAL_SIZE   = 1000
BERT_TEST_SIZE  = 1000

# ── Slice from already-split arrays ──────────────────────────────
X_bert_train = list(X_train[:BERT_TRAIN_SIZE])
y_bert_train = list(y_train[:BERT_TRAIN_SIZE])
X_bert_val   = list(X_val[:BERT_VAL_SIZE])
y_bert_val   = list(y_val[:BERT_VAL_SIZE])
X_bert_test  = list(X_test[:BERT_TEST_SIZE])
y_bert_test  = list(y_test[:BERT_TEST_SIZE])

print(f'BERT train: {len(X_bert_train)} | val: {len(X_bert_val)} | test: {len(X_bert_test)}')

# ── Tokenizer ────────────────────────────────────────────────────
bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

# ── Dataset class ────────────────────────────────────────────────
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

train_dataset = NewsDataset(X_bert_train, y_bert_train, bert_tokenizer, BERT_MAX_LEN)
val_dataset   = NewsDataset(X_bert_val,   y_bert_val,   bert_tokenizer, BERT_MAX_LEN)
test_dataset  = NewsDataset(X_bert_test,  y_bert_test,  bert_tokenizer, BERT_MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BERT_BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BERT_BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BERT_BATCH_SIZE)

# ── Model ────────────────────────────────────────────────────────
bert_model = BertForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2)
bert_model = bert_model.to(device)

# ── Optimizer & scheduler ────────────────────────────────────────
optimizer   = AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps = len(train_loader) * BERT_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print('BERT model ready. Trainable params:',
      sum(p.numel() for p in bert_model.parameters() if p.requires_grad))

## Cell 13 — BERT: Train

In [ ]:
def bert_evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_preds), np.array(all_labels)


bert_train_accs,   bert_val_accs   = [], []
bert_train_losses, bert_val_losses = [], []

for epoch in range(BERT_EPOCHS):
    bert_model.train()
    total_train_loss = 0
    train_preds, train_labels_ep = [], []

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = bert_model(input_ids=input_ids,
                             attention_mask=attention_mask,
                             labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels_ep.extend(labels.cpu().numpy())

        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss {loss.item():.4f}')

    train_acc  = accuracy_score(train_labels_ep, train_preds)
    train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, _, _ = bert_evaluate(bert_model, val_loader, device)

    bert_train_accs.append(train_acc)
    bert_val_accs.append(val_acc)
    bert_train_losses.append(train_loss)
    bert_val_losses.append(val_loss)

    print(f'\nEpoch {epoch+1}/{BERT_EPOCHS}')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Val   Loss: {val_loss:.4f}   | Val   Acc: {val_acc:.4f}\n')

## Cell 14 — BERT: Evaluate & Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, BERT_EPOCHS+1)

axes[0].plot(epochs_range, bert_train_accs, label='Train Acc', marker='o')
axes[0].plot(epochs_range, bert_val_accs,   label='Val Acc',   marker='s')
axes[0].set_title('BERT Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, bert_train_losses, label='Train Loss', marker='o')
axes[1].plot(epochs_range, bert_val_losses,   label='Val Loss',   marker='s')
axes[1].set_title('BERT Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

_, _, bert_preds, bert_true = bert_evaluate(bert_model, test_loader, device)
results.append(evaluate_model('BERT (fine-tuned)', bert_true, bert_preds))

## Cell 15 — Confusion Matrices

In [ ]:
model_preds = [
    ('SVM (TF-IDF)',      y_test,    svm_preds),
    ('Random Forest',     y_test,    rf_preds),
    ('BiLSTM',            y_test,    lstm_preds),
    ('BERT (fine-tuned)', bert_true, bert_preds),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (name, y_true, y_pred) in zip(axes, model_preds):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Fake','Real'], yticklabels=['Fake','Real'])
    acc = accuracy_score(y_true, y_pred)
    ax.set_title(f'{name}\nAcc: {acc:.4f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 16 — Model Comparison

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
print('\n' + '='*60)
print('         FINAL MODEL COMPARISON')
print('='*60)
print(results_df.round(4).to_string())
print('='*60)

metrics = ['accuracy', 'precision', 'recall', 'f1']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

for ax, metric in zip(axes, metrics):
    values = results_df[metric]
    bars = ax.bar(range(len(values)), values.values,
                  color=colors, edgecolor='black', alpha=0.85)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(values.index, rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0.85, 1.01)
    ax.set_title(metric.capitalize(), fontweight='bold')
    ax.set_ylabel('Score'); ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, values.values):
        ax.text(bar.get_x() + bar.get_width()/2, v+0.002,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 17 — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

svm_scores = svm_pipeline.decision_function(X_test)
fpr, tpr, _ = roc_curve(y_test, svm_scores)
ax.plot(fpr, tpr, label=f'SVM (AUC={roc_auc_score(y_test, svm_scores):.3f})', lw=2)

rf_scores = rf_pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, rf_scores)
ax.plot(fpr, tpr, label=f'RF  (AUC={roc_auc_score(y_test, rf_scores):.3f})', lw=2)

fpr, tpr, _ = roc_curve(y_test, lstm_probs)
ax.plot(fpr, tpr, label=f'BiLSTM (AUC={roc_auc_score(y_test, lstm_probs):.3f})', lw=2)

ax.plot([0,1],[0,1],'k--', lw=1.5, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 18 — Error Analysis

In [ ]:
errors_df = pd.DataFrame({
    'text':      list(X_test),
    'true':      y_test,
    'predicted': lstm_preds,
    'prob':      lstm_probs
})
errors = errors_df[errors_df['true'] != errors_df['predicted']]
fn = errors[errors['true'] == 1]
fp = errors[errors['true'] == 0]
print(f'LSTM misclassified: {len(errors)}/{len(errors_df)} ({len(errors)/len(errors_df)*100:.1f}%)')
print(f'  False Negatives (Real→Fake): {len(fn)}')
print(f'  False Positives (Fake→Real): {len(fp)}')
if len(fp):
    print('\nSample FP:', fp.iloc[0]['text'][:300])
if len(fn):
    print('\nSample FN:', fn.iloc[0]['text'][:300])

## Cell 19 — Save Models

In [ ]:
lstm_model.save('lstm_fake_news_model.h5')
print('LSTM saved.')

bert_model.save_pretrained('bert_fake_news_model')
bert_tokenizer.save_pretrained('bert_fake_news_model')
print('BERT saved.')

results_df.to_csv('model_results_summary.csv')
print('Results CSV saved.')

## Cell 20 — Inference Demo

In [ ]:
def predict_lstm(text):
    cleaned = clean_text(text)
    seq     = tokenizer_lstm.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
    prob    = float(lstm_model.predict(padded, verbose=0)[0][0])
    label   = 'REAL' if prob >= 0.5 else 'FAKE'
    print(f'  LSTM → {label}  (conf: {max(prob, 1-prob):.2%})')

def predict_bert(text):
    bert_model.eval()
    enc = bert_tokenizer(
        text, max_length=BERT_MAX_LEN,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        out = bert_model(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        )
    probs = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
    label = 'REAL' if probs[1] >= 0.5 else 'FAKE'
    print(f'  BERT → {label}  (conf: {max(probs):.2%})')

samples = [
    "The Federal Reserve raised interest rates by 0.25 percent, citing persistent inflation.",
    "SHOCKING: Government hiding cure for all diseases! Share before they delete this!!!"
]

for i, text in enumerate(samples, 1):
    print(f'\nSample {i}: "{text[:80]}"')
    predict_lstm(text)
    predict_bert(text)

## Summary

| Cell | What it does |
|------|--------------|
| 1 | Install deps |
| 2 | All imports (AdamW from `torch.optim`) |
| 3 | Load dataset via kagglehub |
| 4 | EDA — distributions, word clouds |
| 5 | Text cleaning pipeline |
| 6 | Sample 20k, train/val/test split |
| 7 | Shared `evaluate_model()` helper |
| 8 | SVM + Random Forest baselines |
| 9 | LSTM tokenize & pad |
| 10 | BiLSTM build & train |
| 11 | LSTM evaluation + training curves |
| 12 | BERT datasets, model, optimizer (all in one) |
| 13 | BERT training loop |
| 14 | BERT evaluation + training curves |
| 15 | Confusion matrices (all 4 models) |
| 16 | Bar chart comparison |
| 17 | ROC curves |
| 18 | Error analysis |
| 19 | Save models |
| 20 | Inference demo |